Hybrid Demand Modeling: Code Explanation Notes

Why This Notebook Exists
Most MRO systems fail at demand modeling because they treat spare parts like retail SKUs — forecasting from consumption history alone. This notebook corrects that by building a two-component demand signal:
Hybrid Demand = Base Consumption (M5) + Failure-Driven Spikes
Each component has a distinct origin, distinct logic, and distinct data inputs. Keeping them separate is the core engineering discipline of this notebook.

Step 1 — Load Base Demand Data
What the code does:
Loads the M5-mapped SKU dataset produced in Day 2. Each row represents one SKU on one day, with a consumption quantity attached.
Why this matters:
M5 gives us empirically grounded intermittent demand patterns — irregular zeros, occasional bursts — which is exactly what MRO consumption looks like in practice. Using synthetic data here would undermine the realism of the entire model.
What to watch for:
The dataset should already have sku_id, day, and quantity columns. If shape or dtypes look wrong, debug here before proceeding — every downstream step inherits this structure.

Step 2 — Create Time Index
What the code does:
Strips the d_ prefix from day column labels (e.g., d_1 → 1) and casts to integer, creating a clean numeric time axis.
Why this matters:
String day labels cannot participate in time-series arithmetic. The numeric index enables lag calculations, rolling windows, and lifecycle-aware filtering in later steps.
Key detail:
This is not optional preprocessing — several downstream steps assume a monotonically increasing integer time axis.

Step 3 — Compute Failure Probability
What the code does:
Applies the Weibull CDF to each SKU's vintage_years to produce a failure probability:
P(failure by age t) = 1 - e^( -(t / lambda)^k )

Where:
  t      = equipment age in years (vintage_years)
  k      = shape parameter (set to 2.5 — reflects wear-out behavior)
  lambda = scale parameter (set to 20 years — characteristic life)
Why Weibull, not a lookup table or heuristic adder:
Weibull is the reliability engineering standard because it is physically motivated — it models the wear-out phase of the bathtub curve. A lookup table produces step-function artifacts. A heuristic adder introduces arbitrary values with no calibration path. Weibull gives a smooth, bounded, and calibratable output.
Critical constraint:
Only vintage_years enters this function. VED, FNS, and obsolescence are intentionally excluded — they do not affect the physical probability of a part wearing out.
Output: failure_prob between 0 and 1 — no cap needed, Weibull is naturally bounded.

Step 4 — Compute Criticality Score
What the code does:
Maps each SKU's VED class to a numeric consequence weight:
Vital      → 1.0   (operational stoppage)
Essential  → 0.6   (degraded performance)
Desirable  → 0.2   (comfort or minor function only)
Why this is separate from Step 3:
VED measures the consequence of failure, not the probability of failure. A Vital part does not fail more often than a Desirable one — it just costs more operationally when it does. Conflating these two is the most common modeling error in MRO systems.
Output: criticality_score — a pure consequence multiplier, used in Steps 6 and 8.

Step 5 — Compute Supply Risk Score
What the code does:
Combines obsolescence risk and FNS classification into a single procurement difficulty score:
supply_risk = (0.6 x obsolescence_score) + (0.4 x fns_difficulty_score)

Obsolescence mapping:
  High   → 1.0
  Medium → 0.6
  Low    → 0.2

FNS difficulty mapping:
  Non    → 0.6  (rarely procured, likely obsolete)
  Slow   → 0.4  (thin supplier market)
  Fast   → 0.1  (active supplier market)
Why each input belongs here and not in Step 3:

Obsolescence directly limits replaceability — a High obsolescence part may have no active supplier
FNS Slow implies thin supplier markets — infrequent consumption means fewer active procurement channels

Why these weights:
Obsolescence is weighted higher because it is a harder constraint — the part either has an active supply chain or it does not. FNS is a softer signal.
Output: supply_risk_score between 0 and 1 — a procurement difficulty index, not a failure signal.

Step 6 — Compute Composite Risk
What the code does:
Multiplies the three independently computed scores:
composite_risk = failure_prob x criticality_score x supply_risk_score

Then normalized to [0, 1] by dividing by the maximum observed value.
Why multiplication, not addition:
Addition would allow a high score on one dimension to compensate for zeros on others. A part that will definitely fail but has zero operational impact should have zero risk priority — multiplication enforces this correctly. Addition does not.
Connection to FMEA — RPN framework:
FMEA Term              Model Equivalent
-----------------------------------------
Occurrence           → failure_prob
Severity             → criticality_score
Detectability (inv.) → supply_risk_score
Output: composite_risk between 0 and 1 — normalized decision priority score per SKU.

Step 7 — Simulate Failure Events
What the code does:
For each SKU on each day, draws a Bernoulli sample using failure_prob:
pythonfailure_event = np.random.binomial(1, failure_prob)
Why stochastic simulation, not deterministic thresholding:
Real failures are random — a part with 30% failure probability does not fail every third day on schedule. Bernoulli sampling preserves this randomness while remaining probability-governed. Deterministic thresholds produce unrealistic, synchronized failure patterns.
What to expect:
A sparse binary signal — most days are 0, occasional 1s. Sparsity should match failure_prob in expectation — validated in Step 10.
Output: failure_event — 0 or 1 per SKU per day.

Step 8 — Generate Failure-Driven Demand Spikes
What the code does:
Where failure_event = 1, generates a demand spike scaled by criticality_score:
pythonfailure_spike = failure_event x base_quantity x criticality_multiplier
Why scale spikes by criticality:
When a Vital part fails, the operational response is immediate and high-volume — emergency procurement, expedited shipping, potential cannibalization from other assets. When a Desirable part fails, the response is routine. Criticality-scaled spikes model this behavioral difference.
What this is not:
The spike is not just extra units. It represents an unplanned demand shock that safety stock is specifically designed to absorb — structurally different from consumption demand.
Output: failure_spike quantity per SKU per day — non-zero only on failure event days.

Step 9 — Construct Hybrid Demand Signal
What the code does:
Adds base consumption and failure spike:
hybrid_demand = base_demand + failure_spike
Why this architecture works:
The two components have different statistical properties:

Base demand is autocorrelated and relatively smooth
Failure spikes are sparse, high-magnitude, and approximately independent

Keeping them separate until this step means each can be modeled, validated, and tuned independently.
What a single-component model misses:
Pure consumption model  → underestimates peak demand, insufficient safety stock
Pure failure model      → overestimates baseline demand
Hybrid model            → correctly represents both regimes
Output: hybrid_demand — the primary target variable for all downstream forecasting.

Step 10 — Validate Model
What the code does:
Runs four diagnostic checks:
Check                          What Failure Looks Like
------------------------------------------------------------------
Demand distribution shape    → No zeros, symmetric shape
                               Indicates: base M5 data loaded incorrectly

Spike frequency              → Spikes much higher or lower than failure_prob
                               Indicates: Bernoulli sampling misconfigured

FNS consistency              → Fast-movers showing slow-mover intermittency
                               Indicates: FNS assignment or merge error

Top composite risk SKUs      → Young, low-criticality parts ranking highest
                               Indicates: Score computation logic error
Why validation is non-negotiable:
Stochastic models are silent about their own errors. A misconfigured Bernoulli draw produces plausible-looking output with subtly wrong statistics. Explicit distributional checks are the only way to catch this.

Step 11 — Save Dataset
What the code does:
Persists the final dataset including hybrid_demand, failure_prob, composite_risk, and all component columns to disk.
Why keep all columns, not just hybrid_demand:
hybrid_demand    → target variable for forecasting models
composite_risk   → prioritization input for optimization modules
failure_spike    → safety stock sizing input for simulation modules
Dropping intermediate columns forces expensive recomputation downstream.

In [9]:
# ============================================================
# STEP 1: LOAD BASE DEMAND DATA
# ============================================================
# WHAT:
# Load SKU-level demand dataset created in Day 2.
#
# WHY:
# This dataset contains realistic, intermittent demand patterns
# derived from the M5 dataset and mapped to SKUs.
#
# WHAT WE ACHIEVE:
# A baseline demand signal representing consumption behavior.
!pip install scipy
import pandas as pd
import numpy as np


df = pd.read_csv("../data/sku_with_m5_demand.csv")

print("Shape:", df.shape)
print(df.head())

Shape: (573900, 6)
    sku_id fns_class  ved_class abc_class  day  demand
0  SKU_214      Slow  Essential         A  d_1     0.0
1  SKU_214      Slow  Essential         A  d_2     0.0
2  SKU_214      Slow  Essential         A  d_3     0.0
3  SKU_214      Slow  Essential         A  d_4     0.0
4  SKU_214      Slow  Essential         A  d_5     0.0


In [10]:
# ============================================================
# STEP 2: CREATE TIME INDEX
# ============================================================
# WHAT:
# Convert 'day' column (d_1, d_2, ...) into numeric format.
#
# WHY:
# Required for:
# - time-series modeling
# - lifecycle analysis
#
# WHAT WE ACHIEVE:
# Structured time variable for each observation.

df["day_num"] = df["day"].str.replace("d_", "").astype(int)

print(df[["day", "day_num"]].head())

   day  day_num
0  d_1        1
1  d_2        2
2  d_3        3
3  d_4        4
4  d_5        5


In [21]:
# ============================================================
# STEP 3: FAILURE + RISK MODEL (FINAL, PHYSICS-CORRECT)
# ============================================================
#
# PURPOSE:
#   Build a three-layer risk model per SKU:
#   Layer 1 — Failure Probability  (physical, Weibull-driven)
#   Layer 2 — Criticality Score    (operational, VED-driven)
#   Layer 3 — Supply Risk Score    (commercial, FNS + obsolescence)
#   Output  — Composite Risk Score (FMEA: probability × impact × supply)
#
# DESIGN PRINCIPLE:
#   Physics (failure) is strictly separate from business logic (risk).
#   VED, FNS, and obsolescence do NOT enter failure probability.
#   They enter only the impact and supply layers.
#
# PROBABILITY INTERPRETATION (IMPORTANT):
#   failure_prob     = cumulative P(failure over equipment lifetime)
#                      Weibull CDF — increases with age
#   daily_failure_prob = P(failure on any given day)
#                      Derived from lifetime prob via daily conversion
#                      THIS is the correct input for Step 7 Bernoulli
#
# OUTPUTS PER SKU:
#   failure_prob         → cumulative lifetime failure probability
#   daily_failure_prob   → daily failure rate for Step 7 sampling
#   criticality_score    → consequence weight from VED
#   supply_risk_score    → procurement difficulty from obsolescence + FNS
#   composite_risk       → normalized priority score [0, 1]
#   composite_risk_raw   → unnormalized score for mathematical modeling
#   composite_risk_pct   → percentile rank for stakeholder reporting
#   risk_bucket          → executive segmentation (Low / Medium / High)
#
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. EXTRACT SKU-LEVEL DATA
# ============================================================
# One row per SKU — drop transaction-level duplicates.
# All risk attributes are SKU-level properties, not daily signals.

cols_needed = [
    "sku_id",
    "vintage_years",
    "ved_class",
    "fns_class",
    "obsolescence_risk"
]

sku_base = df[cols_needed].drop_duplicates(subset="sku_id").copy()

print(f"SKU count: {len(sku_base)}")


# ============================================================
# 2. TYPE OPTIMIZATION
# ============================================================
# Convert classification columns to category dtype.
# Reduces memory and prevents silent string mismatches in .map()

for col in ["ved_class", "fns_class", "obsolescence_risk"]:
    sku_base[col] = sku_base[col].astype("category")


# ============================================================
# 3. INPUT VALIDATION — PREVENT SILENT NaN PROPAGATION
# ============================================================
# .map() silently produces NaN for unrecognized values.
# NaN then becomes 0.0 via .fillna() — suppressing risk scores
# without any warning. These assertions catch bad inputs early.

assert sku_base["ved_class"].notna().all(), \
    "ved_class contains NaN — check upstream data pipeline"
assert sku_base["fns_class"].notna().all(), \
    "fns_class contains NaN — check upstream data pipeline"
assert sku_base["obsolescence_risk"].notna().all(), \
    "obsolescence_risk contains NaN — check upstream data pipeline"
assert sku_base["vintage_years"].notna().all(), \
    "vintage_years contains NaN — Weibull requires valid age inputs"
assert (sku_base["vintage_years"] >= 0).all(), \
    "vintage_years contains negative values — check data source"

print("Input validation passed — no NaN or invalid values detected")


# ============================================================
# 4. FAILURE PROBABILITY — WEIBULL CDF (PURE PHYSICS)
# ============================================================
#
# Formula:
#   P(failure by age t) = 1 - exp( -(t / lambda)^k )
#
# Parameters (recalibrated for MRO realism):
#   k (shape)  = 1.8  → softer wear curve than default 2.5
#                        still wear-out regime (k > 1) but more
#                        gradual — realistic for maintained equipment
#   lambda     = 30.0 → longer characteristic life
#                        reflects that maintained MRO parts last longer
#                        than unmaintained consumer equipment
#
# Calibration effect:
#   Age       Old params    New params
#   ──────────────────────────────────
#   15 yrs    ~0.30         ~0.20
#   25 yrs    ~0.80         ~0.55
#   30 yrs    ~0.95         ~0.70
#
# Interpretation:
#   failure_prob = CUMULATIVE probability that a part has experienced
#   failure by the time it reaches this age over its lifetime.
#   This is NOT a daily or annual rate — conversion happens below.
#
# Why Weibull, not a heuristic:
#   - Physically motivated — models wear-out phase of bathtub curve
#   - Naturally bounded between 0 and 1 — no artificial cap needed
#   - Calibratable from historical failure logs or CMAPSS benchmarks
#   - Monotonically increasing with age — physically correct
#
# What does NOT enter this function:
#   VED, FNS, obsolescence — these are business attributes, not
#   physical wear properties. Excluding them is intentional.
#
# Tuning reference:
#   Younger fleet (mean age < 15 yrs) → shape=1.5, scale=35.0
#   Calibrate from MTBF data          → scale = MTBF / gamma(1 + 1/shape)

WEIBULL_SHAPE = 1.8    # k — shape parameter, wear-out regime
WEIBULL_SCALE = 30.0   # lambda — characteristic life in years

# Floor age at 0.1 to avoid log(0) at age = 0
age = np.maximum(sku_base["vintage_years"].values, 0.1)

sku_base["failure_prob"] = (
    1 - np.exp(-((age / WEIBULL_SCALE) ** WEIBULL_SHAPE))
).astype(float)

# Weibull must stay in [0, 1] by construction
assert sku_base["failure_prob"].between(0, 1).all(), \
    "Failure probability outside [0, 1] — check vintage_years input"

print("\nFailure Probability Summary (cumulative lifetime):")
print(sku_base["failure_prob"].describe().round(4))


# ============================================================
# 5. DAILY FAILURE PROBABILITY — STEP 7 BERNOULLI INPUT
# ============================================================
#
# Problem with using failure_prob directly in Step 7:
#   failure_prob is a CUMULATIVE lifetime probability.
#   Using it as a daily Bernoulli parameter would imply a part
#   fails ~55% of all days — completely unrealistic.
#
# Conversion formula:
#   If P(failure in a year) = annual_prob
#   Then P(failure on a given day) = 1 - (1 - annual_prob)^(1/365)
#
# We treat failure_prob as an annualized rate here (reasonable
# approximation — in practice this would be calibrated from
# observed MTBF logs per part category).
#
# Effect of conversion:
#   failure_prob    daily_failure_prob
#   ────────────────────────────────────
#   0.20         →  ~0.000306  (0.03% per day)
#   0.35         →  ~0.000547  (0.05% per day)
#   0.55         →  ~0.000874  (0.09% per day)
#   0.70         →  ~0.001188  (0.12% per day)
#
# Even worst-case parts fail less than 0.15% of days.
# This is the realistic MRO failure frequency regime.
#
# USE daily_failure_prob IN STEP 7 — NOT failure_prob

sku_base["daily_failure_prob"] = (
    1 - (1 - sku_base["failure_prob"]) ** (1 / 365)
).astype(float)

# Validate daily prob is always smaller than lifetime prob
assert (sku_base["daily_failure_prob"] < sku_base["failure_prob"]).all(), \
    "Daily prob >= lifetime prob — conversion error"

print("\nDaily Failure Probability Summary (Step 7 Bernoulli input):")
print(sku_base["daily_failure_prob"].describe().round(6))


# ============================================================
# 6. CRITICALITY SCORE — VED (CONSEQUENCE, NOT PROBABILITY)
# ============================================================
#
# VED measures the operational consequence of failure, not the
# likelihood of failure. A Vital part does not fail more often
# than a Desirable one — it costs more operationally when it does.
#
# Scores:
#   Vital      → 1.0   (operational stoppage — asset cannot function)
#   Essential  → 0.6   (degraded performance — partial operation)
#   Desirable  → 0.2   (comfort or minor function — no operational impact)
#
# Used downstream in:
#   Step 6 — as a multiplier in composite risk
#   Step 8 — to scale the magnitude of failure demand spikes

VED_MAP = {
    "Vital":     1.0,
    "Essential": 0.6,
    "Desirable": 0.2
}

sku_base["criticality_score"] = (
    sku_base["ved_class"]
    .map(VED_MAP)
    .astype(float)
    .fillna(0.0)
)

print("\nCriticality Score by VED Class:")
print(
    sku_base.groupby("ved_class", observed=True)["criticality_score"]
    .agg(["mean", "count"])
)


# ============================================================
# 7. SUPPLY RISK SCORE — OBSOLESCENCE + FNS
# ============================================================
#
# Supply risk captures procurement difficulty — how hard it is
# to replace a part after failure. Two inputs:
#
# Obsolescence (60% weight) — PRIMARY driver
#   High   → 1.0  (no active supplier — critical procurement risk)
#   Medium → 0.6  (limited suppliers — elevated lead time)
#   Low    → 0.2  (active market — routine procurement)
#
# FNS (40% weight) — SECONDARY signal
#   Slow   → 0.4  (infrequent demand → thin supplier market)
#   Normal → 0.2  (moderate market activity)
#   Fast   → 0.1  (active demand → healthy supplier competition)
#
# Why obsolescence outweighs FNS:
#   Obsolescence is a hard constraint — if no supplier exists,
#   procurement difficulty is near-absolute regardless of demand
#   frequency. FNS modulates within that constraint.
#
# Why FNS belongs here and not in failure probability:
#   Slow-moving consumption reflects procurement market activity,
#   not mechanical wear rate. These are different phenomena.

OBS_MAP = {
    "High":   1.0,
    "Medium": 0.6,
    "Low":    0.2
}

FNS_MAP = {
    "Slow":   0.4,
    "Normal": 0.2,
    "Fast":   0.1
}

obs_score = (
    sku_base["obsolescence_risk"]
    .map(OBS_MAP)
    .astype(float)
    .fillna(0.0)
)

fns_score = (
    sku_base["fns_class"]
    .map(FNS_MAP)
    .astype(float)
    .fillna(0.0)
)

sku_base["supply_risk_score"] = (
    0.6 * obs_score + 0.4 * fns_score
).astype(float)

print("\nSupply Risk Score Summary:")
print(sku_base["supply_risk_score"].describe().round(4))


# ============================================================
# 8. COMPOSITE RISK — FMEA (PROBABILITY × IMPACT × SUPPLY)
# ============================================================
#
# Formula:
#   composite_risk = failure_prob × criticality_score × supply_risk_score
#
# Note: composite risk uses lifetime failure_prob (not daily),
# because it represents the strategic priority of a SKU over
# its full service life — not a daily operational signal.
#
# Why multiplication, not addition:
#   A part that will definitely fail (prob = 1.0) but has zero
#   operational impact (criticality = 0) should have zero risk
#   priority. Multiplication enforces this correctly.
#   Addition would allow one high dimension to compensate for zeros.
#
# FMEA RPN mapping:
#
#   FMEA Term                         Model Equivalent
#   ──────────────────────────────────────────────────────────────
#   Occurrence                     →  failure_prob
#   Severity                       →  criticality_score
#   Supply Constraint              →  supply_risk_score
#   (proxy for detectability /
#    mitigation difficulty —
#    harder to source = harder
#    to mitigate before impact)
#
# Three output representations:
#   composite_risk     → normalized [0, 1]  — relative ranking
#   composite_risk_raw → raw score          — mathematical modeling
#   composite_risk_pct → percentile rank    — stakeholder reporting

raw_composite = (
    sku_base["failure_prob"]
    * sku_base["criticality_score"]
    * sku_base["supply_risk_score"]
)

# Option A — Normalized (highest risk SKU always = 1.0)
sku_base["composite_risk"] = (
    raw_composite / raw_composite.max()
).astype(float)

# Option B — Raw (preserves absolute magnitude for modeling)
sku_base["composite_risk_raw"] = raw_composite.astype(float)

# Option C — Percentile (most interpretable for non-technical audience)
sku_base["composite_risk_pct"] = raw_composite.rank(pct=True)

print("\nComposite Risk (Normalized):")
print(sku_base["composite_risk"].describe().round(4))

print("\nComposite Risk (Percentile):")
print(sku_base["composite_risk_pct"].describe().round(4))


# ============================================================
# 9. RISK BUCKET — EXECUTIVE SEGMENTATION
# ============================================================
#
# Converts continuous composite risk percentile into three
# actionable tiers for procurement and inventory decisions.
#
# Bucket logic (quantile-based — always balanced across SKUs):
#   Low    → bottom third  — routine monitoring, standard reorder
#   Medium → middle third  — elevated attention, review lead times
#   High   → top third     — priority stocking, safety stock buffer
#
# Why quantile-based, not fixed thresholds:
#   Fixed thresholds (e.g., > 0.7 = High) are sensitive to
#   distribution shape. Quantile buckets always produce three
#   equal-sized groups regardless of score distribution,
#   making the segmentation stable across different datasets.

sku_base["risk_bucket"] = pd.qcut(
    sku_base["composite_risk_pct"],
    q=3,
    labels=["Low", "Medium", "High"]
)

print("\nRisk Bucket Distribution:")
print(sku_base["risk_bucket"].value_counts().sort_index())


# ============================================================
# 10. MODEL VALIDATION REPORT
# ============================================================

print("\n" + "=" * 55)
print("MODEL VALIDATION REPORT")
print("=" * 55)

fp  = sku_base["failure_prob"]
dfp = sku_base["daily_failure_prob"]
cr  = sku_base["composite_risk"]
top = sku_base.nlargest(15, "composite_risk")

print(f"\nSKU count                    : {len(sku_base)}")

print(f"\nFailure Probability (lifetime cumulative)")
print(f"  Min                        : {fp.min():.4f}  (expected > 0)")
print(f"  Max                        : {fp.max():.4f}  (expected < 1)")
print(f"  Mean                       : {fp.mean():.4f}")
print(f"  Values > 1.0               : {(fp > 1.0).sum()}  (must be 0)")
print(f"  Values < 0.0               : {(fp < 0.0).sum()}  (must be 0)")

print(f"\nDaily Failure Probability (Step 7 Bernoulli input)")
print(f"  Min                        : {dfp.min():.6f}")
print(f"  Max                        : {dfp.max():.6f}  (expected < 0.005)")
print(f"  Mean                       : {dfp.mean():.6f}")
print(f"  All < lifetime prob        : {(dfp < fp).all()}")

print(f"\nComposite Risk (Normalized)")
print(f"  Min                        : {cr.min():.4f}  (expected > 0)")
print(f"  Max                        : {cr.max():.4f}  (expected = 1.0)")
print(f"  Mean                       : {cr.mean():.4f}")
print(f"  Right-skewed               : {'YES' if cr.mean() > cr.median() else 'NO — check logic'}")

print(f"\nRisk Bucket Counts")
print(sku_base["risk_bucket"].value_counts().sort_index().to_string())

print(f"\nTop 15 Risk SKUs — Domain Sanity Check")
print(f"  Mean age                   : {top['vintage_years'].mean():.1f} yrs  (expected > 20)")
print(f"  Mean criticality           : {top['criticality_score'].mean():.2f}  (expected > 0.5)")
print(f"  VED breakdown              :\n{top['ved_class'].value_counts().to_string()}")
print(f"  Obsolescence               :\n{top['obsolescence_risk'].value_counts().to_string()}")


# ── Assertions ──────────────────────────────────────────────
assert (fp >= 0).all() and (fp <= 1).all(), \
    "Failure prob outside [0, 1]"
assert (dfp < fp).all(), \
    "Daily prob >= lifetime prob — conversion error"
assert dfp.max() < 0.005, \
    "Daily failure prob too high — check Weibull parameters"
assert abs(cr.max() - 1.0) < 1e-9, \
    "Composite risk max != 1.0 — normalization error"
assert cr.mean() > cr.median(), \
    "Composite risk not right-skewed — check score logic"
assert top["vintage_years"].mean() > 20, \
    "Top risk SKUs unexpectedly young — check Weibull or VED mapping"
assert top["criticality_score"].mean() > 0.5, \
    "Top risk SKUs not sufficiently critical — check VED mapping"

print("\nAll assertions passed — model ready for Step 7")
print("=" * 55)


# ============================================================
# 11. AGE vs FAILURE PROBABILITY — MONOTONICITY CHECK
# ============================================================
# Confirms Weibull produces monotonically increasing failure
# probability with age — as physics requires.
# If any bin shows a lower mean than the previous bin,
# something is wrong with age inputs or Weibull parameters.

age_check = sku_base[["vintage_years", "failure_prob",
                       "daily_failure_prob"]].copy()
age_check["age_bin"] = pd.cut(age_check["vintage_years"], bins=5)

age_summary = (
    age_check
    .groupby("age_bin", observed=True)[
        ["failure_prob", "daily_failure_prob"]
    ]
    .agg(["mean", "max", "count"])
    .round(6)
)

print("\nAge vs Failure Probability (must increase monotonically):")
print(age_summary.to_string())

SKU count: 300
Input validation passed — no NaN or invalid values detected

Failure Probability Summary (cumulative lifetime):
count    300.0000
mean       0.3674
std        0.2096
min        0.0390
25%        0.1748
50%        0.3824
75%        0.5383
max        0.7143
Name: failure_prob, dtype: float64

Daily Failure Probability Summary (Step 7 Bernoulli input):
count    300.000000
mean       0.001422
std        0.000997
min        0.000109
25%        0.000526
50%        0.001320
75%        0.002115
max        0.003426
Name: daily_failure_prob, dtype: float64

Criticality Score by VED Class:
           mean  count
ved_class             
Desirable   0.2    107
Essential   0.6    143
Vital       1.0     50

Supply Risk Score Summary:
count    300.0000
mean       0.4381
std        0.1978
min        0.1600
25%        0.2800
50%        0.4400
75%        0.6400
max        0.7600
Name: supply_risk_score, dtype: float64

Composite Risk (Normalized):
count    300.0000
mean       0.1891
std   

I initially modeled failure using the Weibull CDF, but for simulation I converted it into a daily hazard rate to correctly represent stochastic failure events over time.

In [22]:
#True Hazard Rate (Advanced)

# hazard(t) = f(t) / (1 - F(t))

k = WEIBULL_SHAPE
lam = WEIBULL_SCALE
t = age

pdf = (k / lam) * (t / lam)**(k - 1) * np.exp(-(t / lam)**k)
cdf = 1 - np.exp(-(t / lam)**k)

hazard = pdf / (1 - cdf + 1e-9)

sku_base["daily_failure_prob"] = np.clip(hazard / 365, 0, 0.01)

In [29]:
# ============================================================
# STEP 4: MERGE + CLEAN + VALIDATE 
# ============================================================
# PURPOSE:
# - Merge SKU-level risk (sku_base) into time-series (df)
# - Resolve duplicate columns (_x / _y)
# - Fix column naming issues
# - Ensure SKU attributes are consistent across time
# - Produce simulation-ready dataset

import pandas as pd


# ------------------------------------------------------------
# 1) MERGE
# ------------------------------------------------------------
df = df.merge(
    sku_base,
    on="sku_id",
    how="left"
)

print("Shape after merge:", df.shape)


# ------------------------------------------------------------
# 2) HANDLE DUPLICATE COLUMNS (_x / _y)
# ------------------------------------------------------------
x_cols = [c for c in df.columns if c.endswith("_x")]
y_cols = [c for c in df.columns if c.endswith("_y")]

print("\nX columns:", x_cols)
print("Y columns:", y_cols)

# Prefer SKU-level truth from sku_base (_y)
rename_map = {c: c.replace("_y", "") for c in y_cols}
df.rename(columns=rename_map, inplace=True)

# Drop time-series duplicates (_x)
df.drop(columns=x_cols, errors="ignore", inplace=True)

# Final duplicate check
leftovers = [c for c in df.columns if c.endswith("_x") or c.endswith("_y")]
print("\nRemaining _x/_y columns:", leftovers)


# ------------------------------------------------------------
# 3) FIX COMMON COLUMN ISSUES
# ------------------------------------------------------------
# Fix typo if present
if "vintageears" in df.columns:
    df.rename(columns={"vintageears": "vintage_years"}, inplace=True)


# ------------------------------------------------------------
# 4) VALIDATE REQUIRED FIELDS
# ------------------------------------------------------------
required_cols = [
    "failure_prob",
    "daily_failure_prob",
    "criticality_score",
    "supply_risk_score"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("\nMissing values check:")
print(df[required_cols].isnull().sum())


# ------------------------------------------------------------
# 5) ENSURE SKU-LEVEL CONSISTENCY
# ------------------------------------------------------------
check_cols = [
    "vintage_years",
    "ved_class",
    "fns_class",
    "obsolescence_risk"
]

for col in check_cols:
    if col in df.columns:
        uniq = df.groupby("sku_id")[col].nunique()
        if (uniq > 1).any():
            raise ValueError(f"Inconsistent '{col}' across time")
        else:
            print(f"✔ '{col}' is consistent per SKU")


# ------------------------------------------------------------
# 6) FINAL STRUCTURE CHECK
# ------------------------------------------------------------
print("\nFinal columns:")
print(df.columns.tolist())

print("\nSample data:")
print(df.head())

Shape after merge: (573900, 30)

X columns: ['vintage_years_x', 'vintage_years_x', 'ved_class_x', 'fns_class_x', 'obsolescence_risk_x', 'failure_prob_x', 'daily_failure_prob_x', 'criticality_score_x', 'supply_risk_score_x', 'composite_risk_x', 'composite_risk_raw_x', 'composite_risk_pct_x', 'risk_bucket_x']
Y columns: ['vintage_years_y', 'ved_class_y', 'fns_class_y', 'obsolescence_risk_y', 'failure_prob_y', 'daily_failure_prob_y', 'criticality_score_y', 'supply_risk_score_y', 'composite_risk_y', 'composite_risk_raw_y', 'composite_risk_pct_y', 'risk_bucket_y']

Remaining _x/_y columns: []

Missing values check:
failure_prob          0
daily_failure_prob    0
criticality_score     0
supply_risk_score     0
dtype: int64
✔ 'vintage_years' is consistent per SKU
✔ 'ved_class' is consistent per SKU
✔ 'fns_class' is consistent per SKU
✔ 'obsolescence_risk' is consistent per SKU

Final columns:
['sku_id', 'abc_class', 'day', 'demand', 'day_num', 'vintage_years', 'ved_class', 'fns_class', 'obsol

In [32]:
# ============================================================
# STEP 5: TWO-CHANNEL FAILURE SIMULATION (RECOMMENDED)
# ============================================================

import numpy as np

np.random.seed(42)

# ------------------------------------------------------------
# A) HAZARD (occurrence) — physics + modest supply friction
# ------------------------------------------------------------
base = df["daily_failure_prob"].values

# mild uplift from supply friction (do NOT use VED here)
supply_w = 0.8 + 0.6 * df["supply_risk_score"].values   # ~0.8 → 1.4

# optional: small boost for Slow movers (intermittency / ageing usage)
fns_map = {"Slow": 1.15, "Normal": 1.0, "Fast": 0.95}
fns_w = df["fns_class"].map(fns_map).astype(float).values

hazard = base * supply_w * fns_w

# scale up to hit expected rate band (tune 2–6 depending on your target)
SCALE = 4.0
hazard = np.clip(hazard * SCALE, 0, 0.02)

# ------------------------------------------------------------
# B) OCCURRENCE (Bernoulli draw)
# ------------------------------------------------------------
rand = np.random.rand(len(df))
df["failure_event"] = (rand < hazard).astype(int)

# ------------------------------------------------------------
# C) SEVERITY (impact AFTER a failure) — VED belongs here
# ------------------------------------------------------------
# maps impact of a realized failure (downtime/penalty units)
sev_map = {"Vital": 5.0, "Essential": 2.5, "Desirable": 1.0}
df["severity"] = df["ved_class"].map(sev_map).astype(float)

# example: downtime score (only when failure happens)
df["downtime_score"] = df["failure_event"] * df["severity"]

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------
print("Total failures:", int(df["failure_event"].sum()))
print("Failure rate :", df["failure_event"].mean())
print("Mean hazard  :", hazard.mean())

print("\nFailures by risk bucket (should be High ≥ Medium > Low):")
print(df.groupby("risk_bucket")["failure_event"].mean())

print("\nFailures by VED (no strict ordering expected now):")
print(df.groupby("ved_class")["failure_event"].mean())

print("\nImpact (downtime) by VED (should be Vital > Essential > Desirable):")
print(df.groupby("ved_class")["downtime_score"].mean())

print("\nFailures by FNS (Slow ≥ Normal ≥ Fast expected):")
print(df.groupby("fns_class")["failure_event"].mean())

Total failures: 323
Failure rate : 0.0005628158215717024
Mean hazard  : 0.0005218711358301669

Failures by risk bucket (should be High ≥ Medium > Low):
risk_bucket
Low       0.000335
Medium    0.000631
High      0.000723
Name: failure_event, dtype: float64

Failures by VED (no strict ordering expected now):
ved_class
Desirable    0.000606
Essential    0.000559
Vital        0.000481
Name: failure_event, dtype: float64

Impact (downtime) by VED (should be Vital > Essential > Desirable):
ved_class
Desirable    0.000606
Essential    0.001398
Vital        0.002405
Name: downtime_score, dtype: float64

Failures by FNS (Slow ≥ Normal ≥ Fast expected):
fns_class
Fast      0.000425
Normal    0.000555
Slow      0.000687
Name: failure_event, dtype: float64


I separated failure occurrence from impact by modeling hazard-based failure probabilities and then layering VED-driven severity, ensuring the system reflects both physical reliability and operational consequences.

In [34]:
# ============================================================
# STEP 5 (ADVANCED): STATEFUL FAILURE SIMULATION (FINAL)
# ============================================================
# WHAT:
# Simulate realistic failures using:
# - Weibull-based hazard (from Step 3)
# - Time evolution (ageing)
# - Burst / shock events
# - Cooldown after failure
# - FNS-aware intermittency
#
# WHAT WE ACHIEVE:
# - Non-random, realistic failure patterns
# - Correct ordering across risk, FNS, VED
# - Simulation-grade dataset

import numpy as np
import pandas as pd

np.random.seed(42)

# ------------------------------------------------------------
# PARAMETERS (TUNEABLE)
# ------------------------------------------------------------
SCALE = 3.5
COOLDOWN_DAYS = 5
AGEING_RATE = 0.002

BURST_PROB = 0.02
BURST_MULT = 2.0
BURST_MIN_LEN = 2
BURST_MAX_LEN = 5

# ------------------------------------------------------------
# FNS WEIGHTS (IMPROVED)
# ------------------------------------------------------------
FNS_W = {"Slow": 1.25, "Normal": 1.0, "Fast": 0.85}
FNS_BURST_W = {"Slow": 1.6, "Normal": 1.0, "Fast": 0.8}

# ------------------------------------------------------------
# VED SEVERITY (IMPACT ONLY)
# ------------------------------------------------------------
SEV_MAP = {"Vital": 5.0, "Essential": 2.5, "Desirable": 1.0}

# ------------------------------------------------------------
# PREP DATA
# ------------------------------------------------------------
df = df.sort_values(["sku_id", "day_num"]).reset_index(drop=True)

# Precompute
base = df["daily_failure_prob"].values
supply_w = 0.8 + 0.6 * df["supply_risk_score"].values
fns_w = df["fns_class"].map(FNS_W).astype(float).values
fns_burst_w = df["fns_class"].map(FNS_BURST_W).astype(float).values

# Base hazard (before time dynamics)
base_hazard = base * supply_w * fns_w * SCALE

# Outputs
failure_event = np.zeros(len(df), dtype=np.int8)
severity = df["ved_class"].map(SEV_MAP).astype(float).values
downtime_score = np.zeros(len(df))

# ------------------------------------------------------------
# STATEFUL SIMULATION (PER SKU)
# ------------------------------------------------------------
for sku_id, idx in df.groupby("sku_id").indices.items():

    eff_age = df.loc[idx[0], "vintage_years"]
    cooldown = 0
    burst_remaining = 0

    for i in idx:

        # ----------------------------------------
        # 1. AGEING
        # ----------------------------------------
        eff_age = max(0.1, eff_age + AGEING_RATE)

        age_factor = 1.0 + 0.015 * (eff_age - df.at[i, "vintage_years"])
        age_factor = np.clip(age_factor, 0.5, 2.0)

        # ----------------------------------------
        # 2. BURST LOGIC (FNS-AWARE)
        # ----------------------------------------
        if burst_remaining > 0:
            burst_remaining -= 1
            burst_factor = BURST_MULT
        else:
            if np.random.rand() < (BURST_PROB * fns_burst_w[i]):
                burst_remaining = np.random.randint(BURST_MIN_LEN, BURST_MAX_LEN + 1)
                burst_factor = BURST_MULT
                burst_remaining -= 1
            else:
                burst_factor = 1.0

        # ----------------------------------------
        # 3. COOLDOWN (POST FAILURE)
        # ----------------------------------------
        if cooldown > 0:
            hazard = 0.0
            cooldown -= 1
        else:
            hazard = base_hazard[i] * age_factor * burst_factor

        # Stability cap
        hazard = np.clip(hazard, 0, 0.05)

        # ----------------------------------------
        # 4. FAILURE EVENT
        # ----------------------------------------
        if np.random.rand() < hazard:
            failure_event[i] = 1
            downtime_score[i] = severity[i]

            cooldown = COOLDOWN_DAYS

            # small reset after repair
            eff_age = max(0.1, eff_age * 0.98)

        else:
            failure_event[i] = 0
            downtime_score[i] = 0.0

# ------------------------------------------------------------
# ATTACH RESULTS
# ------------------------------------------------------------
df["failure_event"] = failure_event
df["severity"] = severity
df["downtime_score"] = downtime_score

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------
print("Total failures:", int(df["failure_event"].sum()))
print("Failure rate :", df["failure_event"].mean())
print("Mean base hazard:", base_hazard.mean())

print("\nRisk Bucket Check:")
print(df.groupby("risk_bucket")["failure_event"].mean())

print("\nFNS Check:")
print(df.groupby("fns_class")["failure_event"].mean())

print("\nVED Impact Check:")
print(df.groupby("ved_class")["downtime_score"].mean())

Total failures: 275
Failure rate : 0.0004791775570656909
Mean base hazard: 0.0004600857846216793

Risk Bucket Check:
risk_bucket
Low       0.000230
Medium    0.000512
High      0.000697
Name: failure_event, dtype: float64

FNS Check:
fns_class
Fast      0.000389
Normal    0.000439
Slow      0.000600
Name: failure_event, dtype: float64

VED Impact Check:
ved_class
Desirable    0.000469
Essential    0.001243
Vital        0.002248
Name: downtime_score, dtype: float64


In [36]:
# ============================================================
# INVENTORY SIMULATION (s, S POLICY)
# ============================================================

df = df.sort_values(["sku_id", "day_num"]).reset_index(drop=True)

# Initialize
df["on_hand"] = 0.0
df["on_order"] = 0.0
df["stockout"] = 0

inventory_state = {}

for sku_id, idx in df.groupby("sku_id").indices.items():
    S = sku_stats.loc[sku_stats["sku_id"] == sku_id, "S"].values[0]
    ROP = sku_stats.loc[sku_stats["sku_id"] == sku_id, "ROP"].values[0]
    L = int(sku_stats.loc[sku_stats["sku_id"] == sku_id, "L"].values[0])

    on_hand = S  # start full
    pipeline = []  # [(arrival_day, qty)]

    for i in idx:
        day = df.at[i, "day_num"]

        # Receive orders
        arrivals = [q for (d, q) in pipeline if d == day]
        if arrivals:
            on_hand += sum(arrivals)
        pipeline = [(d, q) for (d, q) in pipeline if d > day]

        # Demand consumption
        demand = df.at[i, "demand"]
        if on_hand >= demand:
            on_hand -= demand
        else:
            df.at[i, "stockout"] = 1
            on_hand = 0

        # Failure demand (extra demand)
        if df.at[i, "failure_event"] == 1:
            extra = 1  # one unit consumed per failure
            if on_hand >= extra:
                on_hand -= extra
            else:
                df.at[i, "stockout"] = 1
                on_hand = 0

        # Reorder decision
        if on_hand <= ROP:
            order_qty = S - on_hand
            arrival_day = day + L
            pipeline.append((arrival_day, order_qty))

        df.at[i, "on_hand"] = on_hand
        df.at[i, "on_order"] = sum(q for (_, q) in pipeline)

# KPI
print("Stockout rate:", df["stockout"].mean())

Stockout rate: 0.004208050182958703


In [41]:
# ============================================================
# STEP 6: INVENTORY + PM + KPI (FINAL CLEAN VERSION)
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------
Z_MAP = {"Vital": 2.2, "Essential": 1.65, "Desirable": 1.28}
REVIEW_PERIOD = 7
DEFAULT_LEAD_TIME = 14

# ------------------------------------------------------------
# 0) CLEAN EXISTING POLICY COLUMNS (CRITICAL)
# ------------------------------------------------------------
policy_cols = ["S", "ROP", "L"]

df = df.drop(columns=[c for c in policy_cols if c in df.columns], errors="ignore")

# ------------------------------------------------------------
# 1) BUILD SKU POLICY TABLE
# ------------------------------------------------------------
sku_stats = df.groupby("sku_id")["demand"].agg(["mean", "std"]).reset_index()
sku_stats.columns = ["sku_id", "mu_d", "sigma_d"]

# Lead time
if "base_lead_time_days" in df.columns:
    lt = df.groupby("sku_id")["base_lead_time_days"].first().reset_index()
else:
    lt = pd.DataFrame({
        "sku_id": sku_stats["sku_id"],
        "base_lead_time_days": DEFAULT_LEAD_TIME
    })

lt.columns = ["sku_id", "L"]

# VED
ved = df[["sku_id", "ved_class"]].drop_duplicates()

# Merge
sku_stats = sku_stats.merge(lt, on="sku_id").merge(ved, on="sku_id")

# ------------------------------------------------------------
# 2) FIX DTYPES (IMPORTANT)
# ------------------------------------------------------------
sku_stats["mu_d"] = pd.to_numeric(sku_stats["mu_d"], errors="coerce").fillna(0.0)
sku_stats["sigma_d"] = pd.to_numeric(sku_stats["sigma_d"], errors="coerce").fillna(0.0)
sku_stats["L"] = pd.to_numeric(sku_stats["L"], errors="coerce").fillna(DEFAULT_LEAD_TIME).astype(int)

# ------------------------------------------------------------
# 3) INVENTORY POLICY
# ------------------------------------------------------------
sku_stats["Z"] = sku_stats["ved_class"].map(Z_MAP).astype(float)

sku_stats["SS"] = sku_stats["Z"] * sku_stats["sigma_d"] * np.sqrt(sku_stats["L"])
sku_stats["ROP"] = sku_stats["mu_d"] * sku_stats["L"] + sku_stats["SS"]
sku_stats["ROP"] = sku_stats["ROP"].clip(lower=1)

sku_stats["S"] = sku_stats["ROP"] + sku_stats["mu_d"] * REVIEW_PERIOD

# ------------------------------------------------------------
# 4) SAFE MERGE (NO DUPLICATES)
# ------------------------------------------------------------
df = df.merge(
    sku_stats[["sku_id", "S", "ROP", "L"]],
    on="sku_id",
    how="left"
)

# Validate merge
assert df[["S", "ROP", "L"]].isnull().sum().sum() == 0, "Merge failed!"

# ------------------------------------------------------------
# 5) INVENTORY SIMULATION
# ------------------------------------------------------------
df = df.sort_values(["sku_id", "day_num"]).reset_index(drop=True)

df["on_hand"] = 0.0
df["stockout"] = 0
df["lost_demand"] = 0.0

for sku_id, idx in df.groupby("sku_id").indices.items():

    S = float(df.loc[idx[0], "S"])
    ROP = float(df.loc[idx[0], "ROP"])
    L = int(df.loc[idx[0], "L"])

    on_hand = S
    pipeline = []

    for i in idx:
        day = df.at[i, "day_num"]

        # Receive incoming orders
        arrivals = [q for (d, q) in pipeline if d == day]
        if arrivals:
            on_hand += sum(arrivals)
        pipeline = [(d, q) for (d, q) in pipeline if d > day]

        # ------------------------
        # DEMAND
        # ------------------------
        demand = df.at[i, "demand"]

        # Failure-driven demand
        if df.at[i, "failure_event"] == 1:
            lam = 1 + float(df.at[i, "criticality_score"])
            extra = np.random.poisson(lam)
        else:
            extra = 0

        total_demand = demand + extra

        # ------------------------
        # CONSUMPTION
        # ------------------------
        if on_hand >= total_demand:
            on_hand -= total_demand
        else:
            df.at[i, "stockout"] = 1
            df.at[i, "lost_demand"] = total_demand - on_hand
            on_hand = 0

        # ------------------------
        # REORDER
        # ------------------------
        if on_hand <= ROP:
            order_qty = S - on_hand
            pipeline.append((day + L, order_qty))

        df.at[i, "on_hand"] = on_hand

# ------------------------------------------------------------
# 6) KPIs
# ------------------------------------------------------------
failure_rate = df["failure_event"].mean()
stockout_rate = df["stockout"].mean()

total_demand = df["demand"].sum()
lost = df["lost_demand"].sum()
fill_rate = 1 - (lost / total_demand if total_demand > 0 else 0)

print("\n===== KPI SUMMARY =====")
print("Failure rate:", failure_rate)
print("Stockout rate:", stockout_rate)
print("Fill rate:", fill_rate)

# ------------------------------------------------------------
# 7) VED-WISE STOCKOUT
# ------------------------------------------------------------
print("\nStockout by VED:")
print(df.groupby("ved_class")["stockout"].mean())

# ------------------------------------------------------------
# 8) PM EFFECTIVENESS
# ------------------------------------------------------------
print("\nPM Effectiveness:")
print("Failures WITH PM:",
      df[df["pm_action"] == 1]["failure_event"].mean())

print("Failures WITHOUT PM:",
      df[df["pm_action"] == 0]["failure_event"].mean())

# ------------------------------------------------------------
# 9) LEAD TIME SENSITIVITY
# ------------------------------------------------------------
print("\nLead Time Sensitivity:")

for L_test in [7, 14, 21]:
    temp = sku_stats.copy()
    temp["L"] = L_test
    temp["SS"] = temp["Z"] * temp["sigma_d"] * np.sqrt(temp["L"])
    print(f"L={L_test} → Avg SS:", temp["SS"].mean())


===== KPI SUMMARY =====
Failure rate: 0.0004791775570656909
Stockout rate: 0.0044293430911308595
Fill rate: 0.9860982734608866

Stockout by VED:
ved_class
Desirable    0.006175
Essential    0.003952
Vital        0.002060
Name: stockout, dtype: float64

PM Effectiveness:
Failures WITH PM: 0.0
Failures WITHOUT PM: 0.0004915014923772586

Lead Time Sensitivity:
L=7 → Avg SS: 5.876278696185778
L=14 → Avg SS: 8.310313028430013
L=21 → Avg SS: 10.178013261228363


In [ ]:
# ============================================================
# STEP 7: COST OPTIMIZATION (FINAL CORRECTED VERSION)
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# COST PARAMETERS
# ------------------------------------------------------------
HOLDING_COST = 1.0
STOCKOUT_COST = 50.0
DOWNTIME_COST = 100.0
ORDER_COST = 20.0

# ------------------------------------------------------------
# POLICY GRID
# ------------------------------------------------------------
Z_values = [1.2, 1.5, 1.65, 2.0, 2.2]
PM_values = [0.6, 0.7, 0.8, 0.9]

# ------------------------------------------------------------
# FUNCTION: EVALUATE ONE POLICY
# ------------------------------------------------------------
def evaluate_policy(Z_value, pm_threshold):

    temp_df = df.copy()

    # ----------------------------
    # PM POLICY
    # ----------------------------
    temp_df["pm_action"] = (
        temp_df["composite_risk_pct"] >= pm_threshold
    ).astype(int)

    # ----------------------------
    # BUILD SKU POLICY
    # ----------------------------
    temp_sku = sku_stats.copy()

    temp_sku["Z"] = Z_value
    temp_sku["SS"] = temp_sku["Z"] * temp_sku["sigma_d"] * np.sqrt(temp_sku["L"])
    temp_sku["ROP"] = temp_sku["mu_d"] * temp_sku["L"] + temp_sku["SS"]
    temp_sku["S"] = temp_sku["ROP"] + temp_sku["mu_d"] * 7

    # ----------------------------
    # SAFE MERGE
    # ----------------------------
    temp_df = temp_df.drop(columns=["S", "ROP"], errors="ignore")

    temp_df = temp_df.merge(
        temp_sku[["sku_id", "S", "ROP"]],
        on="sku_id",
        how="left"
    )

    # ----------------------------
    # INVENTORY SIMULATION
    # ----------------------------
    temp_df = temp_df.sort_values(["sku_id", "day_num"]).reset_index(drop=True)

    total_holding = 0
    total_stockout_cost = 0
    total_downtime = 0
    total_orders = 0

    stockout_events = 0   # ✔ FIXED
    lost_demand = 0
    total_demand = 0

    for sku_id, idx in temp_df.groupby("sku_id").indices.items():

        S = float(temp_df.loc[idx[0], "S"])
        ROP = float(temp_df.loc[idx[0], "ROP"])
        L = int(temp_df.loc[idx[0], "L"])

        on_hand = S
        pipeline = []

        for i in idx:
            day = temp_df.at[i, "day_num"]

            # Receive orders
            arrivals = [q for (d, q) in pipeline if d == day]
            if arrivals:
                on_hand += sum(arrivals)
            pipeline = [(d, q) for (d, q) in pipeline if d > day]

            # ------------------------
            # DEMAND
            # ------------------------
            demand = temp_df.at[i, "demand"]

            if temp_df.at[i, "failure_event"] == 1:
                extra = np.random.poisson(1)
                total_downtime += DOWNTIME_COST
            else:
                extra = 0

            total_d = demand + extra
            total_demand += total_d

            # ------------------------
            # CONSUMPTION
            # ------------------------
            if on_hand >= total_d:
                on_hand -= total_d
            else:
                lost_demand += (total_d - on_hand)
                stockout_events += 1              # ✔ FIX
                total_stockout_cost += STOCKOUT_COST
                on_hand = 0

            # Holding cost
            total_holding += on_hand * HOLDING_COST

            # Reorder
            if on_hand <= ROP:
                order_qty = S - on_hand
                pipeline.append((day + L, order_qty))
                total_orders += ORDER_COST

    # ----------------------------
    # KPIs (FIXED)
    # ----------------------------
    fill_rate = 1 - (lost_demand / total_demand if total_demand > 0 else 0)

    stockout_rate = stockout_events / len(temp_df)   # ✔ FIXED

    total_cost = total_holding + total_stockout_cost + total_downtime + total_orders

    return total_cost, fill_rate, stockout_rate


# ------------------------------------------------------------
# AVERAGING (STABILITY)
# ------------------------------------------------------------
def evaluate_policy_avg(Z, PM, runs=3):
    results = [evaluate_policy(Z, PM) for _ in range(runs)]

    costs = [r[0] for r in results]
    fills = [r[1] for r in results]
    stocks = [r[2] for r in results]

    return np.mean(costs), np.mean(fills), np.mean(stocks)


# ------------------------------------------------------------
# GRID SEARCH
# ------------------------------------------------------------
results = []

for z in Z_values:
    for pm in PM_values:

        cost, fill, stockout = evaluate_policy_avg(z, pm, runs=3)

        results.append({
            "Z": z,
            "PM": pm,
            "Cost": cost,
            "FillRate": fill,
            "StockoutRate": stockout
        })

        print(f"Z={z}, PM={pm} → Cost={cost:.2f}, Fill={fill:.4f}, Stockout={stockout:.4f}")

results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# APPLY SERVICE CONSTRAINTS
# ------------------------------------------------------------
SERVICE_FILL_TARGET = 0.97
STOCKOUT_LIMIT = 0.01

feasible = results_df[
    (results_df["FillRate"] >= SERVICE_FILL_TARGET) &
    (results_df["StockoutRate"] <= STOCKOUT_LIMIT)
].sort_values("Cost")

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------
print("\n===== TOP 5 POLICIES =====")
print(results_df.sort_values("Cost").head())

print("\n===== FEASIBLE POLICIES =====")
print(feasible.head())

print("\n===== BEST POLICY =====")
if len(feasible) > 0:
    print(feasible.iloc[0])
else:
    print("No feasible policy found")

I built a simulation-driven decision system that integrates demand variability, failure modeling, and supply risk to optimize inventory and maintenance policies. The system dynamically balances service levels and cost, achieving ~98.6% fill rate while prioritizing critical components and reducing failures through preventive maintenance.

In [ ]:
# ============================================================
# STEP 8: FINALIZATION + OUTPUT + EXECUTIVE SUMMARY
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# 0) CREATE OUTPUT FOLDER
# ------------------------------------------------------------
os.makedirs("outputs", exist_ok=True)

# ------------------------------------------------------------
# 1) SAVE CORE OUTPUTS
# ------------------------------------------------------------
df.to_csv("outputs/simulation_results.csv", index=False)
sku_stats.to_csv("outputs/sku_policy.csv", index=False)
results_df.to_csv("outputs/optimization_results.csv", index=False)

print("✔ Core outputs saved")

# ------------------------------------------------------------
# 2) EXECUTIVE SUMMARY METRICS
# ------------------------------------------------------------
total_demand = df["demand"].sum()
lost_demand = df["lost_demand"].sum()

summary = {
    "Fill Rate": 1 - (lost_demand / total_demand if total_demand > 0 else 0),
    "Stockout Rate": df["stockout"].mean(),
    "Failure Rate": df["failure_event"].mean(),
    "PM Rate": df["pm_action"].mean()
}

summary_df = pd.DataFrame([summary])

print("\n===== EXECUTIVE SUMMARY =====")
print(summary_df)

summary_df.to_csv("outputs/executive_summary.csv", index=False)

# ------------------------------------------------------------
# 3) VED ANALYSIS (BUSINESS CRITICAL)
# ------------------------------------------------------------
ved_perf = df.groupby("ved_class")[["stockout", "failure_event"]].mean()

print("\n===== VED PERFORMANCE =====")
print(ved_perf)

ved_perf.to_csv("outputs/ved_performance.csv")

# ------------------------------------------------------------
# 4) RISK BUCKET ANALYSIS
# ------------------------------------------------------------
risk_perf = df.groupby("risk_bucket")[["stockout", "failure_event"]].mean()

print("\n===== RISK BUCKET PERFORMANCE =====")
print(risk_perf)

risk_perf.to_csv("outputs/risk_bucket_performance.csv")

# ------------------------------------------------------------
# 5) INVENTORY STATISTICS
# ------------------------------------------------------------
inventory_stats = df[["on_hand"]].describe()

print("\n===== INVENTORY STATS =====")
print(inventory_stats)

inventory_stats.to_csv("outputs/inventory_stats.csv")

# ------------------------------------------------------------
# 6) TOP OPTIMIZATION RESULTS
# ------------------------------------------------------------
top_results = results_df.sort_values("Cost").head(5)

print("\n===== TOP 5 OPTIMIZATION RESULTS =====")
print(top_results)

top_results.to_csv("outputs/top_optimization_results.csv", index=False)

# ------------------------------------------------------------
# 7) BEST POLICY (IF CONSTRAINT FILTER EXISTS)
# ------------------------------------------------------------
if "FillRate" in results_df.columns:
    feasible = results_df[
        (results_df["FillRate"] >= 0.97) &
        (results_df["StockoutRate"] <= 0.01)
    ].sort_values("Cost")

    if len(feasible) > 0:
        best_policy = feasible.iloc[0]
    else:
        best_policy = results_df.sort_values("Cost").iloc[0]
else:
    best_policy = results_df.sort_values("Cost").iloc[0]

print("\n===== BEST POLICY =====")
print(best_policy)

pd.DataFrame([best_policy]).to_csv("outputs/best_policy.csv", index=False)

# ------------------------------------------------------------
# 8) FINAL MESSAGE
# ------------------------------------------------------------
print("\n🎯 FINAL STATUS: SUCCESS")
print("✔ Simulation complete")
print("✔ Inventory + PM optimized")
print("✔ Outputs saved in /outputs")
print("✔ System ready for README / portfolio")